In [1]:
pip install sentence-transformers scikit-learn

   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---------------------------------------- 588.9/588.9 kB 4.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [4]:
df = pd.read_csv(
    "zuno_dataset.csv"
)

df.head()

,song,artist,genre,popularity,lyrics
0,i'm yours,jason mraz,acoustic,80,Well you done done me and you bet I felt it ...
1,i won't give up,jason mraz,acoustic,69,When I look into your eyes It's like watchi...
2,93 million miles,jason mraz,acoustic,0,93 million miles from the sun People get re...
3,bella luna,jason mraz,acoustic,1,Mystery the moon A hole in the sky A sup...
4,winter wonderland,jason mraz,acoustic,0,"Sleigh bells ring, are you listening, In th..."


In [6]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
df['combined_text'] = (
    df['song'].astype(str)
    + " "
    + df['artist'].astype(str)
    + " "
    + df['genre'].astype(str)
    + " "
    + df['lyrics'].astype(str)
)

In [26]:
df_small=df.copy()

In [27]:
embeddings = model.encode(
    df_small['combined_text'].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

In [28]:
embeddings.shape

(1360, 384)

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
def recommend(song_name):

    matches = df_small[
        df_small['song']
        .str.contains(
            song_name,
            case=False,
            na=False
        )
    ]

    if len(matches) == 0:

        print("Song Not Found")

        return

    idx = matches.index[0]

    similarity_scores = cosine_similarity(
        [embeddings[idx]],
        embeddings
    )[0]

    top_indices = (
        similarity_scores
        .argsort()[-6:-1]
    )[::-1]

    return df_small.iloc[
        top_indices
    ][[
        'song',
        'artist',
        'genre'
    ]]

In [13]:
df_small[['song']].sample(20)

,song
103,castle of glass
202,castles made of sand
813,little paper boy
93,in bloom
282,time in a bottle
837,every day is exactly the same
588,no excuses
139,merry christmas baby
710,falling to pieces
404,together forever


In [14]:
recommend(
    df_small.iloc[0]['song']
)

,song,artist,genre
5,if it kills me,jason mraz,acoustic
14,"on love, in sadness",jason mraz,acoustic
1,i won't give up,jason mraz,acoustic
725,the flame,cheap trick,hard-rock
288,breathe,faith hill,country


In [15]:
favorite_songs = [

    df_small.iloc[0]['song'],

    df_small.iloc[1]['song'],

    df_small.iloc[2]['song']

]

In [16]:
user_embeddings = []

In [17]:
for song in favorite_songs:

    idx = df_small[
        df_small['song']
        ==
        song
    ].index[0]

    user_embeddings.append(
        embeddings[idx]
    )

In [18]:
user_vector = np.mean(
    user_embeddings,
    axis=0
)

In [19]:
scores = cosine_similarity(
    [user_vector],
    embeddings
)[0]

In [20]:
df_small['score'] = scores

In [21]:
playlist = df_small.sort_values(
    'score',
    ascending=False
).head(20)

In [22]:
playlist[
[
'song',
'artist',
'genre'
]
]

,song,artist,genre
1,i won't give up,jason mraz,acoustic
0,i'm yours,jason mraz,acoustic
2,93 million miles,jason mraz,acoustic
27,long drive,jason mraz,acoustic
235,i'll be waiting,adele,british
907,always,bon jovi,metal
10,no stopping us,jason mraz,acoustic
8,you and i both,jason mraz,acoustic
293,it will rain,bruno mars,dance
5,if it kills me,jason mraz,acoustic


In [23]:
import numpy as np

np.save(
    "zuno_embeddings.npy",
    embeddings
)

print("Embeddings Saved")

Embeddings Saved


In [24]:
embeddings = np.load(
    "zuno_embeddings.npy"
)

In [25]:
playlist.to_csv(
    "recommended_playlist.csv",
    index=False
)